# Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_percentage_error

df = pd.read_csv('Gold  Prices.csv')
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.sort_values('Date').reset_index(drop=True)

print("Dimensi Data Awal:", df.shape)
df.head()

Matplotlib is building the font cache; this may take a moment.


Dimensi Data Awal: (1258, 9)


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains
0,2019-11-11 05:00:00+00:00,137.610001,137.789993,136.440002,137.059998,7037300,0.0,0.0,0.0
1,2019-11-12 05:00:00+00:00,137.029999,137.589996,136.190002,137.429993,6448600,0.0,0.0,0.0
2,2019-11-13 05:00:00+00:00,137.800003,138.220001,137.639999,137.979996,8776000,0.0,0.0,0.0
3,2019-11-14 05:00:00+00:00,138.389999,138.940002,137.869995,138.559998,5220500,0.0,0.0,0.0
4,2019-11-15 05:00:00+00:00,138.029999,138.419998,137.970001,138.210007,10106700,0.0,0.0,0.0


# Feature Engineering

Membuat indikator teknis tambahan seperti return harian (Daily Return), Moving Average harian (MA5 & MA20), rata-rata volume (Vol_MA5), dan volatilitas harga untuk memperkaya informasi model.

In [ ]:
df['Return'] = df['Close'].pct_change()
df['MA5'] = df['Close'].rolling(window=5).mean()
df['MA20'] = df['Close'].rolling(window=20).mean()
df['Vol_MA5'] = df['Volume'].rolling(window=5).mean()
df['Volatility'] = df['Return'].rolling(window=5).std()

df['Target_Close'] = df['Close'].shift(-1)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

feature_cols = [  'Open',
    'High',
    'Low',
    'Close',
    'Volume',
    'Return',
    'MA5',
    'MA20',
    'Vol_MA5',
    'Volatility'
]
X_raw = df[feature_cols].values
y_raw = df['Target_Close'].values


X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

df.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains,Return,MA5,MA20,Vol_MA5,Volatility,Target_Close
0,2020-03-30 04:00:00+00:00,152.410004,153.080002,151.570007,152.919998,12080900,0.0,0.0,0.0,0.004401,152.622000,149.691999,14376880.0,0.024250,148.050003
1,2020-03-31 04:00:00+00:00,151.360001,151.800003,147.970001,148.050003,13319500,0.0,0.0,0.0,-0.031847,151.552002,149.399999,12892080.0,0.017214,149.449997
2,2020-04-01 04:00:00+00:00,148.199997,150.080002,147.850006,149.449997,11827400,0.0,0.0,0.0,0.009456,151.182001,149.164499,11966120.0,0.018042,151.899994
3,2020-04-02 04:00:00+00:00,151.199997,152.500000,150.699997,151.899994,9188300,0.0,0.0,0.0,0.016393,150.913998,148.884998,11248140.0,0.018845,152.649994
4,2020-04-03 04:00:00+00:00,152.229996,153.089996,151.660004,152.649994,8469100,0.0,0.0,0.0,0.004937,150.993997,148.639998,10977040.0,0.018801,156.880005
